In [1]:
from pathlib import Path
import os
import re
import sys
import warnings

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "Data").exists() and (candidate / "Note").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("`Data`와 `Note` 디렉토리를 포함한 프로젝트 루트를 찾지 못했습니다.")

mpl_dir = PROJECT_ROOT / ".mplconfig"
mpl_dir.mkdir(exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_dir)
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.options.display.float_format = "{:,.4f}".format
sns.set_theme(style="whitegrid")
plt.rcParams["axes.unicode_minus"] = False
try:
    plt.rcParams["font.family"] = "AppleGothic"
except Exception:
    pass

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Optuna version: {optuna.__version__}")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /Users/restitutor/Documents/Restitutor/Workspace/Flutter-Python/Python/EP_cycle_stations
Python executable: /usr/local/bin/python3
Optuna version: 4.7.0


In [3]:
import pandas as pd
df = pd.read_csv("/Users/restitutor/Documents/Restitutor/Workspace/Flutter-Python/Python/EP_cycle_stations/Data/Restitutor/resti_dataset_3.csv")

In [4]:
df.head()

,station_id,month,date,hour,weekday,is_weekday,is_restingday,business_ratio,residential_ratio,transit_ratio,...,총 대여수,timestamp,온도,습도,불쾌지수,강수량,적설량,rain_flag,snow_flag,leisure_ratio_sq
0,ST-1331,1,1,0,0,0,1,0.017799,0.015954,0.089229,...,0,2024-01-01 00:00:00,-2.7,92.0,28.49784,0.0,0.0,0,0,0.000293
1,ST-1331,1,1,1,0,0,1,0.017799,0.015954,0.089229,...,0,2024-01-01 01:00:00,-1.2,87.0,31.85344,0.0,0.0,0,0,0.000293
2,ST-1331,1,1,2,0,0,1,0.017799,0.015954,0.089229,...,3,2024-01-01 02:00:00,-1.2,88.0,31.69856,0.0,0.0,0,0,0.000293
3,ST-1331,1,1,3,0,0,1,0.017799,0.015954,0.089229,...,1,2024-01-01 03:00:00,-1.1,88.0,31.86668,0.0,0.0,0,0,0.000293
4,ST-1331,1,1,4,0,0,1,0.017799,0.015954,0.089229,...,2,2024-01-01 04:00:00,-1.0,87.0,32.18770,0.0,0.0,0,0,0.000293
